# 05 · Module 5: Equity return prediction

**Question.** Does ML forecast these 15 stocks well enough to beat a naive forecast?

Expect it to match equal-weight at best; the module earns marks by proving that cleanly.

**Data.** `nova_market_data.csv`: 15 stocks × 1,457 days (Jan 2021 – Aug 2026). Return is same-day close-to-close. Split: monthly walk-forward, expanding window, 21-day gap (`splits.walk_forward_months`).

**Professor's anchor.** S11 (predict, rank, select, allocate, rebalance), S15–S16 (factor zoo, test checklist), TA11 (the look-ahead bug that inflated a Sharpe ratio).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("05")

In [ ]:
market = load("market", parse_dates=["Date"]).sort_values(["Ticker", "Date"]).reset_index(drop=True)
market.head()

## 1. Target

Forward 21-day return: the sum of the next 21 daily Returns, shifted so nothing from t+1 onwards is a feature.

## 2. Features at t only

Momentum_20D, Volatility_20D, 60-day momentum, distance from the 50-day average, volume ratio, sector, VIX, Market_Return lagged one day, the macro series, and each stock's cross-sectional rank on each (ranks within the date, never over the full sample).

## 3. Models

Historical mean (naive baseline), ridge regression, random forest and XGBoost.

## 4. Walk-forward

Train on an expanding window, predict the next month, retrain monthly, 21-day gap.

## 5. Metrics

Out-of-sample R² against the historical mean (`evaluate.oos_r2`), monthly rank IC with its t-statistic (`evaluate.rank_ic`, `evaluate.ic_summary`) and hit rate.

## Handoff to Module 9

Save the out-of-sample predictions to this module's results folder for notebook 06. The portfolio handoff is written in notebook 06.

In [ ]:
# lines = [handoff.make_line(
#     business_line="...", capacity_cr=..., expected_return_pct=..., expected_loss_pct=...,
#     capital_charge_pct=..., stress_loss_pct=..., ml_edge="validated | weak | none", evidence="...",
#     key_risks=[("risk", "control"), ("risk", "control")],
# )]
# handoff.write_handoff("05", lines)

## Traps to check before handing over

Using same-day Return as the target; computing ranks over the full sample; reporting gross Sharpe only.